In [1]:
!pip install pymupdf scikit-learn


In [2]:
import json

sample_persona = {
    "persona": "PhD Researcher in Computational Biology",
    "job_to_be_done": "Prepare a literature review on methodologies and benchmarks in drug discovery"
}

with open("persona.json", "w") as f:
    json.dump(sample_persona, f, indent=2)

print("✓ persona.json created")


✓ persona.json created


In [3]:
import os
import fitz  # PyMuPDF

def extract_sections(pdf_path):
    doc = fitz.open(pdf_path)
    sections = []

    for page_number, page in enumerate(doc, start=1):
        text_blocks = page.get_text("blocks")
        page_text = "\n".join([b[4] for b in text_blocks if b[4].strip() != ""])
        sections.append({
            "document": os.path.basename(pdf_path),
            "page": page_number,
            "text": page_text
        })

    return sections


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def rank_sections(sections, persona_query):
    docs = [s["text"] for s in sections]
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform([persona_query] + docs)

    query_vec = tfidf_matrix[0]
    doc_vecs = tfidf_matrix[1:]

    scores = cosine_similarity(query_vec, doc_vecs).flatten()

    for i, sec in enumerate(sections):
        sec["score"] = scores[i]
        sec["section_title"] = sec["text"][:80] + "..."
        sec["refined_text"] = sec["text"][:500]

    top_sections = sorted(sections, key=lambda x: x["score"], reverse=True)[:5]
    return top_sections


In [5]:
from datetime import datetime

def load_persona(persona_path="persona.json"):
    with open(persona_path, "r") as f:
        data = json.load(f)
    return data["persona"], data["job_to_be_done"]

def generate_output(top_sections, persona, job):
    output = {
        "metadata": {
            "documents": list(set([s["document"] for s in top_sections])),
            "persona": persona,
            "job_to_be_done": job,
            "timestamp": datetime.utcnow().isoformat() + "Z"
        },
        "sections": [
            {
                "document": s["document"],
                "page": s["page"],
                "section_title": s["section_title"],
                "importance_rank": i + 1
            } for i, s in enumerate(top_sections)
        ],
        "sub_sections": [
            {
                "document": s["document"],
                "page": s["page"],
                "refined_text": s["refined_text"],
                "importance_rank": i + 1
            } for i, s in enumerate(top_sections)
        ]
    }

    os.makedirs("output", exist_ok=True)
    with open("output/persona_analysis.json", "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2)
    
    print("✓ Output saved to output/persona_analysis.json")


In [6]:
input_dir = "input"  # make sure PDFs are in this folder
persona_file = "persona.json"

persona, job = load_persona(persona_file)
query = f"{persona} - {job}"

all_sections = []

for filename in os.listdir(input_dir):
    if filename.endswith(".pdf"):
        pdf_path = os.path.join(input_dir, filename)
        sections = extract_sections(pdf_path)
        all_sections.extend(sections)

top_sections = rank_sections(all_sections, query)
generate_output(top_sections, persona, job)


✓ Output saved to output/persona_analysis.json
